# Fine-tuning Lapa LLM for Ukrainian Dialect Translation

This notebook fine-tunes [Lapa LLM v0.1.2 Instruct](https://huggingface.co/lapa-llm/lapa-v0.1.2-instruct) on the `merged.csv` dataset for translating Ukrainian dialects to Standard Literary Ukrainian.

**Model**: Lapa LLM is based on Gemma-3-12B with an optimized tokenizer for Ukrainian, making it 1.5x faster for Ukrainian text processing.

**Task**: Translate dialectal Ukrainian text (Hutsul, Lemko, etc.) to Standard Literary Ukrainian.

## Load Lapa LLM Model

Using Unsloth's `FastLanguageModel` to load the Lapa LLM model with 4-bit quantization for memory efficiency.

In [ ]:
from unsloth import FastModel
import torch

# Load Lapa LLM - Ukrainian-optimized model based on Gemma-3-12B
model, tokenizer = FastModel.from_pretrained(
    model_name = "./test-model",
    max_seq_length = 2048,  # Can adjust for longer sequences
    load_in_4bit = True,    # 4-bit quantization for memory efficiency
    dtype = None,           # Auto-detect dtype
    # token = "YOUR_HF_TOKEN",  # Add if needed for gated models
)

## Prepare LoRA Configuration

Configure LoRA (Low-Rank Adaptation) for efficient fine-tuning. This allows updating only a small subset of parameters.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                    # LoRA rank - higher = more capacity but slower
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,           # LoRA scaling factor
    lora_dropout = 0,          # Dropout (0 = none)
    bias = "none",             # Don't update biases
    use_gradient_checkpointing = "unsloth",  # Enable for longer sequences
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

## Load and Prepare Dataset

Load the `merged.csv` dataset and format it for instruction fine-tuning.

In [ ]:
import pandas as pd
from datasets import Dataset

# Load dataset
df = pd.read_csv('../../data/parallel/merged.csv')

# Preview the data
print(f"Dataset size: {len(df)} samples")
print("\nFirst few examples:")
print(df.head())

# Expected columns: 'dialect_text' and 'standard_text' (adjust if needed)
print("\nColumns:", df.columns.tolist())

In [ ]:
# Format dataset for instruction tuning
# Adjust column names if they're different in your CSV

def format_prompt(dialect_text, standard_text):
    """Format as instruction-following task"""
    return {
        "text": f"""<start_of_turn>user
Переклади наступний діалектний текст на стандартну літературну українську мову:

{dialect_text}<end_of_turn>
<start_of_turn>model
{standard_text}<end_of_turn>"""
    }

# Apply formatting
# Adjust 'dialect_text' and 'standard_text' to match your actual column names
formatted_data = [
    format_prompt(row['dialect_text'], row['standard_text'])
    for _, row in df.iterrows()
]

# Convert to HuggingFace Dataset
dataset = Dataset.from_list(formatted_data)

print(f"\nFormatted dataset size: {len(dataset)}")
print("\nExample formatted prompt:")
print(dataset[0]['text'])

## Configure Training

Set up training parameters using Unsloth's optimized trainer.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,  # Can enable for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,  # Adjust based on dataset size
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",  # Change to "wandb" if using W&B
    ),
)

## Train the Model

Start fine-tuning. This may take a while depending on dataset size and GPU.

In [ ]:
# Show GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
# Train!
trainer_stats = trainer.train()

In [ ]:
# Show memory usage after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## Test the Model

Try inference with the fine-tuned model.

In [ ]:
# Enable fast inference mode
FastLanguageModel.for_inference(model)

# Test with a dialectal example
test_dialect = "Your test dialect text here"  # Replace with actual test text

inputs = tokenizer(
    f"""<start_of_turn>user
Переклади наступний діалектний текст на стандартну літературну українську мову:

{test_dialect}<end_of_turn>
<start_of_turn>model
""",
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    use_cache=True
)

result = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("\n=== Translation Result ===")
print(result)

## Save the Model

Save LoRA adapters, upload to HuggingFace, or export in various formats.

### Save LoRA Adapters Locally

In [ ]:
# Save LoRA adapters (lightweight, ~200MB)
model.save_pretrained("lapa_dialect_lora")
tokenizer.save_pretrained("lapa_dialect_lora")
print("Saved LoRA adapters to 'lapa_dialect_lora'")

### Upload to HuggingFace Hub

In [ ]:
# Upload to HuggingFace (requires token)
if False:  # Set to True to upload
    HF_TOKEN = "YOUR_HF_TOKEN"
    HF_REPO = "YOUR_USERNAME/lapa-dialect-translation-lora"

    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"Pushed to {HF_REPO}")

### Save Merged Model (Full FP16)

In [ ]:
# Save merged model for deployment (requires more disk space)
if False:  # Set to True to save merged model
    model.save_pretrained_merged("lapa_dialect_full", tokenizer)
    print("Saved merged model to 'lapa_dialect_full'")

### Save as GGUF for llama.cpp

In [ ]:
# Save as GGUF for llama.cpp deployment
if False:  # Set to True to save GGUF
    model.save_pretrained_gguf(
        "lapa_dialect_gguf",
        tokenizer,
        quantization_method="q4_k_m",  # Good balance of size/quality
    )
    print("Saved GGUF model to 'lapa_dialect_gguf'")

## Load Saved Model

To load the saved LoRA adapters for inference:

In [ ]:
if False:  # Set to True to load saved model
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="lapa_dialect_lora",
        max_seq_length=2048,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    print("Loaded model from 'lapa_dialect_lora'")